# Mixed Local-Evidence Cross-Dataset Evaluation

Train local-evidence classifiers on mixed Kaggle, Outfox, and MAGE data, then evaluate transfer on each dataset separately for Gemma 3 1B and SmolLM 360M.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

from local_evidence_models_utils import (
    BayesianConfig,
    GaussianMixtureNBConfig,
    LOCAL_FEATURE_COLUMNS,
    MODEL_NAMES,
    run_local_evidence_model_comparison,
    training_input_fingerprint,
)
from local_evidence_utils import (
    default_local_evidence_path,
    load_or_build_local_evidence_windows,
    split_local_evidence_texts,
)

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

In [ ]:
DATASETS = {
    "kaggle_train": "Kaggle",
    "outfox_chatgpt_train": "Outfox",
    "mage_train": "MAGE",
}
MODELS = ["gemma_3_1b_it", "smollm2_360m"]
WINDOW_LENGTHS = (32, 64, 128)
STRIDE = 16
TOP_K = 20
SEED = 42
RECOMPUTE_WINDOWS = False
REUSE_FITTED_MODELS = True
SAVE_FITTED_MODELS = False
TRUST_FITTED_MODEL_CACHE = False
MIXED_REGIME = "Mixed"
BAYESIAN_CONFIG = BayesianConfig(advi_iterations=10_000, posterior_draws=1_000, random_seed=SEED)
GAUSSIAN_MIXTURE_CONFIG = GaussianMixtureNBConfig(components=2, reg_covar=1e-6)
try:
    import pymc
except ModuleNotFoundError:
    PYMC_AVAILABLE = False
else:
    PYMC_AVAILABLE = True
if not PYMC_AVAILABLE:
    display("PyMC unavailable; continuing without pymc_bayesian_logistic. Install it with: uv sync --extra bayesian")
REQUESTED_MODELS = ("logistic_regression", "gaussian_nb", "gaussian_mixture_nb", "pymc_bayesian_logistic")
AVAILABLE_MODELS = tuple(name for name in REQUESTED_MODELS if name != "pymc_bayesian_logistic" or PYMC_AVAILABLE)

In [ ]:
windows_by_model = {}
partitions_by_model = {}
coverage_frames = []
for processing_model in MODELS:
    model_windows = []
    model_partitions = {}
    for dataset, display_dataset in DATASETS.items():
        try:
            windows = load_or_build_local_evidence_windows(
                PROCESSED_ROOT,
                dataset,
                processing_model,
                path=default_local_evidence_path(PROCESSED_ROOT, dataset, processing_model),
                window_lengths=WINDOW_LENGTHS,
                stride=STRIDE,
                top_k=TOP_K,
                recompute=RECOMPUTE_WINDOWS,
            )
            if windows.empty:
                raise ValueError("local-evidence cache is empty")
            windows = windows.copy()
            windows["source_dataset"] = display_dataset
            windows["source_text_id"] = windows["text_id"]
            windows["text_id"] = display_dataset + ":" + windows["source_text_id"].astype(str)
            dataset_partitions = split_local_evidence_texts(windows, seed=SEED)
        except (FileNotFoundError, ValueError) as error:
            display(f"Skipping {display_dataset}/{processing_model}: {error}")
            continue
        model_windows.append(windows)
        model_partitions[display_dataset] = dataset_partitions
        coverage_frames.append(
            windows.groupby("split").size().rename("rows").reset_index().assign(
                processing_model=processing_model, source_dataset=display_dataset
            )
        )
    if model_windows:
        windows_by_model[processing_model] = pd.concat(model_windows, ignore_index=True)
        partitions_by_model[processing_model] = model_partitions
coverage = pd.concat(coverage_frames, ignore_index=True) if coverage_frames else pd.DataFrame()
display(coverage.sort_values(["processing_model", "source_dataset", "split"]))

In [ ]:
result_frames = []
for processing_model, mixed_windows in windows_by_model.items():
    valid_datasets = list(partitions_by_model[processing_model])
    if not valid_datasets:
        continue
    mixed_train = pd.concat(
        [partitions_by_model[processing_model][dataset]["train"] for dataset in valid_datasets],
        ignore_index=True,
    )
    mixed_validation = pd.concat(
        [partitions_by_model[processing_model][dataset]["validation"] for dataset in valid_datasets],
        ignore_index=True,
    )
    training_fingerprint = training_input_fingerprint(
        mixed_windows, {"train": mixed_train, "validation": mixed_validation}
    )
    fitted_models = {}
    for evaluation_dataset in valid_datasets:
        evaluation_partitions = {
            "train": mixed_train,
            "validation": mixed_validation,
            "test": partitions_by_model[processing_model][evaluation_dataset]["test"],
        }
        cache_manifest = {
            "experiment": "mixed_local_evidence_cross_dataset",
            "processing_model": processing_model,
            "datasets": tuple(valid_datasets),
            "features": LOCAL_FEATURE_COLUMNS,
            "window_lengths": WINDOW_LENGTHS,
            "model_names": AVAILABLE_MODELS,
            "bayesian_config": asdict(BAYESIAN_CONFIG),
            "gaussian_mixture_config": asdict(GAUSSIAN_MIXTURE_CONFIG),
            "training_input_fingerprint": training_fingerprint,
        }
        comparison_summary, _, _, _ = run_local_evidence_model_comparison(
            mixed_windows,
            evaluation_partitions,
            AVAILABLE_MODELS,
            feature_columns=LOCAL_FEATURE_COLUMNS,
            seed=SEED,
            bayesian_config=BAYESIAN_CONFIG,
            gaussian_mixture_config=GAUSSIAN_MIXTURE_CONFIG,
            fitted_models=fitted_models,
            cache_root=PROCESSED_ROOT / "cross_dataset_local_evidence" / processing_model / "models",
            cache_manifest=cache_manifest,
            reuse_fitted_models=REUSE_FITTED_MODELS,
            save_fitted_models=SAVE_FITTED_MODELS,
            trust_fitted_models=TRUST_FITTED_MODEL_CACHE,
        )
        result_frames.append(
            comparison_summary.assign(
                processing_model=processing_model,
                evaluation_dataset=evaluation_dataset,
                train_rows=len(mixed_train),
                eval_rows=len(evaluation_partitions["test"]),
            ).rename(columns={"model": "classifier"})
        )
if not result_frames:
    raise ValueError("No mixed local-evidence training or evaluation results are available.")
results = pd.concat(result_frames, ignore_index=True)
display(results.sort_values(["processing_model", "classifier", "window_length", "evaluation_dataset"]))

In [ ]:
for processing_model in MODELS:
    model_results = results.query(
        "processing_model == @processing_model and aggregation == 'evidence_fraction'"
    ).copy()
    if model_results.empty:
        continue
    matrix = (
        model_results.pivot(index=["classifier", "window_length"], columns="evaluation_dataset", values="macro_f1")
        .reindex(columns=["Kaggle", "Outfox", "MAGE"])
    )
    display(matrix)
    plt.figure(figsize=(7, max(3, len(matrix) * 0.45)), constrained_layout=True)
    sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0.0, vmax=1.0, cmap="viridis", mask=matrix.isna())
    plt.title(f"Mixed-training macro-F1 by evaluation dataset: {processing_model}")
    plt.xlabel("Evaluation dataset")
    plt.ylabel("Classifier / window length")
    plt.show()